# Collected edits

All three dermatologists go into one codebook per model, one row per case. A case marked wrong by anyone is edited; if the corrections disagree it is printed and dropped. Reliability, text generality, and image generality run first. Locality is fourth: the cases nobody marked wrong.

`dermedit` kernel. `OVERWRITE` recomputes saved runs. `RESUME` finishes a run that is still missing a metric. With both off, a complete result is printed from disk and an unfinished one stays blank. `REJECT_THRESHOLD` is the key-distance percentile. `run.py` runs both embedding modes at 1 and at 5.

In [1]:
import sys
from pathlib import Path

PERTURB = Path("/scratch/jq2uw/MME/derm_vlm_edit/editing/perturb_generality")
sys.path.insert(0, str(PERTURB))
from utils.bootstrap import bootstrap, results_dir
bootstrap()

import config as C

USERS = [
    "jesse.y.liu.md@gmail.com",
    "achiou@stanford.edu",
    "lhunterellul",
]
MODELS = ["medgemma", "dermatollama"]
RATIONALE_MODE = C.RATIONALE_MODE
NORM_ANSWER = "y16"
SUBSAMPLE = 0
MAX_NEW_TOKENS = C.MAX_NEW_TOKENS
SEED = 42
OVERWRITE = False
RESUME = False

# EMBED_MODE:
#   standalone_dual  DINOv2 image concatenated with SBERT text. This is the current run.
#   dual_sbert       VLM vision activation concatenated with SBERT. Still dual.
#   vision           VLM vision activation only.
#   language         A middle language-layer activation only.
#   language_last    Late language-layer activation. The old VLM-key style.
EMBED_MODE = "language_last"
EMBED_MODES = ["standalone_dual", "language_last"]
# The table lists every percentile. REJECT_THRESHOLD is the one this session computes.
THRESHOLDS = [0.1, 1, 5, 10]
REJECT_THRESHOLD = 0.1
IMAGE_ENCODER = "dinov2"
IMAGE_POOL = "cls"
TEXT_ENCODER = "sbert"
SCALER_DATASET = "coco"

OUT = results_dir(PERTURB, "collect", str(EMBED_MODE) + "_" + str(REJECT_THRESHOLD))
POOL = "all"
print("results:", OUT)

results: /sfs/weka/scratch/jq2uw/MME/derm_vlm_edit/results/perturb_gen/language_last_0.1/collect


In [2]:
import gc

from IPython.display import Markdown, display

from utils.edit import edit
from utils.edits import collect_all, collect_locality
from utils.eval import load_suite, load_vlm, needs_compute, render_table, run_suite

rows = []
for model_short in MODELS:
    if not needs_compute(OUT, POOL, model_short, overwrite=OVERWRITE, resume=RESUME):
        print(f"{POOL} / {model_short}: loaded")
        rows.append(load_suite(OUT, POOL, model_short))
        continue

    edits, report = collect_all(
        USERS, model_short, rationale_mode=RATIONALE_MODE,
        norm_answer=NORM_ANSWER, subsample=SUBSAMPLE,
    )
    print(f"{model_short}: {len(edits)} unique edits, {len(report['conflicts'])} conflicting cases dropped")
    for row in report["conflicts"]:
        users = ", ".join(str(u).split("@")[0] for u in row["users"])
        print(f"  drop {row['case_id']}: {users} -> {', '.join(row['answers'])}")
    locality = collect_locality(USERS, model_short, subsample=SUBSAMPLE)
    print(f"{model_short}: {len(locality)} locality cases")
    if not edits:
        print(f"skip {model_short}")
        continue

    import torch
    vlm, config = load_vlm(
        model_short, max_new_tokens=MAX_NEW_TOKENS, seed=SEED,
        embed_mode=EMBED_MODE, image_encoder=IMAGE_ENCODER,
        image_pool=IMAGE_POOL, text_encoder=TEXT_ENCODER,
        scaler_dataset=SCALER_DATASET, reject_threshold=REJECT_THRESHOLD,
    )
    editor, queries = edit(vlm, config, edits)
    rows.append(run_suite(
        editor, vlm, queries, user=POOL, model_short=model_short,
        out_dir=OUT, locality_edits=locality, seed=SEED,
        norm_answer=NORM_ANSWER, max_new_tokens=MAX_NEW_TOKENS,
        overwrite=OVERWRITE,
    ))
    del vlm, editor
    gc.collect()
    torch.cuda.empty_cache()

saved = []
for mode in EMBED_MODES:
    for pct in THRESHOLDS:
        out = results_dir(PERTURB, "collect", f"{mode}_{pct:g}")
        for model_short in MODELS:
            row = load_suite(out, POOL, model_short)
            row["embed_mode"] = mode
            row["threshold"] = f"{pct:g}"
            saved.append(row)
display(Markdown(render_table(saved)))

all / medgemma: loaded
all / dermatollama: loaded


| Mode | Threshold | Model | Reliability | Text generality | Image generality | Locality |
| --- | --- | --- | --- | --- | --- | --- |
| standalone_dual | 0.1 | MedGemma | 50/52 | 17/52 | 51/52 | 90/171 |
| standalone_dual | 0.1 | DermatoLlama | 32/33 | 12/30 | 29/30 | 101/184 |
| standalone_dual | 1 | MedGemma | 50/50 | 16/50 | 50/50 | 90/171 |
| standalone_dual | 1 | DermatoLlama | 30/30 | 13/30 | 29/30 | 103/184 |
| standalone_dual | 5 | MedGemma | 50/50 | 37/50 | 50/50 | 52/171 |
| standalone_dual | 5 | DermatoLlama | 30/30 | 15/30 | 30/30 | 94/184 |
| standalone_dual | 10 | MedGemma | 50/50 | 41/50 | 50/50 | 34/171 |
| standalone_dual | 10 | DermatoLlama | 30/30 | 24/30 | 30/30 | 64/184 |
| language_last | 0.1 | MedGemma | 50/50 | 48/50 | 50/50 | 71/171 |
| language_last | 0.1 | DermatoLlama | -- | -- | -- | -- |
| language_last | 1 | MedGemma | 50/50 | 48/50 | 50/50 | 71/171 |
| language_last | 1 | DermatoLlama | -- | -- | -- | -- |
| language_last | 5 | MedGemma | 50/50 | 48/50 | 50/50 | 39/171 |
| language_last | 5 | DermatoLlama | -- | -- | -- | -- |
| language_last | 10 | MedGemma | 50/50 | 48/50 | 50/50 | 24/171 |
| language_last | 10 | DermatoLlama | -- | -- | -- | -- |